# HIMSETU · TimesFM-3 stock forecast

Forecasts daily consumption of **diesel, food, medical kits and spares** at Maitri and Bharati with
[TimesFM 3.0](https://github.com/google-research/timesfm), turns it into a stock burn-down, and loads it
into HIMSETU's `forecasts` table (`source = "timesfm"`). The `/inventory` burn-down chart then shows
TimesFM-3 next to the live Holt-Winters line and uses its stock-out date.

**Round trip**

| Step | Cloud Convex deployment | Local `npx convex dev` backend |
|---|---|---|
| 1. Get history | set `CONVEX_URL` below | `npm run forecast:export` → upload `notebooks/history.json` |
| 2. Forecast | run all cells (GPU runtime is faster, CPU works) | same |
| 3. Import | automatic when `PUSH_TO_CONVEX = True` | download `timesfm_forecast.json` → `npm run forecast:import -- timesfm_forecast.json` |

Re-run after **Reset demo**: it regenerates the simulated history and clears imported forecasts.

> Consumption history is **simulated** demo data. TimesFM 3.0 weights are released under Google's
> non-commercial licence: fine for this prototype, not for production use.

In [ ]:
# Install TimesFM 3.0 with the PyTorch backend (about a minute on Colab).
%pip install -q "timesfm[torch]" matplotlib

In [ ]:
# ---- Settings -------------------------------------------------------------
CONVEX_URL = ""                 # e.g. "https://happy-otter-123.convex.cloud"; empty = use HISTORY_FILE
HISTORY_FILE = "history.json"   # from `npm run forecast:export`; Colab asks you to upload it
PUSH_TO_CONVEX = True           # with CONVEX_URL set, write the forecast straight back
OUT_FILE = "timesfm_forecast.json"
MODEL_ID = "google/timesfm-3.0-pytorch"
BACKTEST_DAYS = 14              # hold out the last two weeks to check accuracy

In [ ]:
import json, os, time, urllib.error, urllib.request
import numpy as np

DAY = 86_400_000  # ms; HIMSETU stores every time as UTC milliseconds


def convex(kind, path, args):
    # Call a public Convex function over its HTTP API (kind = "query" | "mutation").
    body = json.dumps({"path": path, "args": args, "format": "json"}).encode()
    req = urllib.request.Request(f"{CONVEX_URL.rstrip('/')}/api/{kind}", data=body,
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            res = json.load(r)
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
    if res.get("status") != "success":
        raise RuntimeError(f"{path}: {res.get('errorMessage', res)}")
    return res["value"]


def read_json(path):
    # PowerShell `>` writes UTF-16, editors add BOMs: accept all of them.
    raw = open(path, "rb").read()
    return json.loads(raw.decode("utf-16" if raw[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig"))


if CONVEX_URL:
    history = convex("query", "forecast:history", {})
else:
    if not os.path.exists(HISTORY_FILE):
        from google.colab import files  # outside Colab, put the file next to the notebook
        HISTORY_FILE = next(iter(files.upload()))
    history = read_json(HISTORY_FILE)

TODAY, HORIZON = history["today"], history["horizon"]


def daily(s):
    # Sort, then put on a gap-free daily grid; missing days are interpolated, not zero.
    d = np.asarray(s["dates"], dtype=np.int64)
    v = np.asarray(s["values"], dtype=np.float64)
    order = np.argsort(d)
    d, v = d[order], v[order]
    grid = np.arange(d[0], d[-1] + DAY, DAY)
    return grid, np.interp(grid, d, v)


series = [s for s in history["series"] if len(s["values"]) >= 2 * 7]
for s in series:
    s["grid"], s["y"] = daily(s)
skipped = len(history["series"]) - len(series)
print(f"{len(series)} series, {len(series[0]['y'])} days each, forecast from "
      f"{time.strftime('%d %b %Y', time.gmtime(TODAY / 1000))} for {HORIZON} days"
      + (f" ({skipped} skipped: under two weeks of history)" if skipped else ""))
for s in series:
    print(f"  {s['stationCode']:<8} {s['item']:<8} stock {s['stock']:>9,.0f} {s['unit']:<5} "
          f"safety {s['safetyLevel']:>8,.0f}  mean use {s['y'].mean():>8,.1f}/day")

In [ ]:
import torch
from timesfm3 import ModelConfig, TimesFM3Evaluator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = TimesFM3Evaluator(ModelConfig(checkpoint_path=MODEL_ID, per_core_batch_size=32, device=DEVICE))
print(f"Loaded {MODEL_ID} on {DEVICE}")


def forecast(contexts, horizon):
    # Point forecast (n, h) and quantiles 0.1..0.9 (n, h, 9); consumption is never negative.
    outs = list(model.predict_batch([np.asarray(c, dtype=np.float32) for c in contexts], horizon=horizon,
                                    return_quantiles=True, use_symmetric_averaging=False))
    return (np.clip(np.stack([o.forecast for o in outs]), 0, None),
            np.clip(np.stack([o.quantiles for o in outs]), 0, None))

### Backtest
Hide the last `BACKTEST_DAYS`, forecast them, and compare with a seasonal-naive baseline (repeat last
week). Skill > 0 means TimesFM-3 beats the baseline.

In [ ]:
ctx = [s["y"][:-BACKTEST_DAYS] for s in series]
truth = np.stack([s["y"][-BACKTEST_DAYS:] for s in series])
pred, _ = forecast(ctx, BACKTEST_DAYS)
naive = np.stack([np.resize(c[-7:], BACKTEST_DAYS) for c in ctx])

mae_tfm = np.abs(pred - truth).mean(axis=1)
mae_naive = np.abs(naive - truth).mean(axis=1)
print(f"{'series':<18}{'MAE TimesFM':>13}{'MAE naive':>11}{'skill':>8}")
for s, a, b in zip(series, mae_tfm, mae_naive):
    skill = 1 - a / b if b else 0.0
    print(f"{s['stationCode'] + '/' + s['item']:<18}{a:>13,.2f}{b:>11,.2f}{skill:>8.0%}")
print(f"Mean skill vs naive: {np.mean([1 - a / b for a, b in zip(mae_tfm, mae_naive) if b]):.0%}")

### Forecast and burn-down
Same rules as the live Holt-Winters line: start at today's stock, subtract forecast use each day,
floor at 0, and the stock-out date is the first day below the safety level. The 0.1 / 0.9 quantiles
give an early/late range for that date.

In [ ]:
point, quant = forecast([s["y"] for s in series], HORIZON)


def burn_down(stock, burn):
    levels = np.maximum(0, stock - np.concatenate([[0.0], np.cumsum(burn)]))
    return [{"date": int(TODAY + i * DAY), "qty": round(float(q), 2)} for i, q in enumerate(levels)]


def stock_out(points, safety):
    return next((p["date"] for p in points if p["qty"] < safety), None)


def fmt(ts):
    return time.strftime("%d %b", time.gmtime(ts / 1000)) if ts else "-"


rows, summary = [], []
for s, p, q in zip(series, point, quant):
    pts = burn_down(s["stock"], p)
    out = stock_out(pts, s["safetyLevel"])
    early = stock_out(burn_down(s["stock"], q[:, -1]), s["safetyLevel"])  # 0.9 quantile: heavy use
    late = stock_out(burn_down(s["stock"], q[:, 0]), s["safetyLevel"])    # 0.1 quantile: light use
    rows.append({"stationCode": s["stationCode"], "item": s["item"], "points": pts,
                 **({"stockOutDate": out} if out else {})})
    summary.append({"stationCode": s["stationCode"], "item": s["item"], "stockOutDate": out,
                    "earliest": early, "latest": late})

print(f"{'series':<18}{'stock-out':>10}{'range (q0.9-q0.1)':>20}")
for r in summary:
    rng = f"{fmt(r['earliest'])} - {fmt(r['latest']) if r['latest'] else f'>{HORIZON} d'}" if r["stockOutDate"] else ""
    print(f"{r['stationCode'] + '/' + r['item']:<18}{fmt(r['stockOutDate']) if r['stockOutDate'] else 'none':>10}{rng:>20}")

In [ ]:
import matplotlib.pyplot as plt

cols = 4
fig, axes = plt.subplots(-(-len(series) // cols), cols, figsize=(16, 3.2 * -(-len(series) // cols)), squeeze=False)
days_fwd = np.arange(1, HORIZON + 1)
for ax, s, p, q in zip(axes.flat, series, point, quant):
    past = (s["grid"] - TODAY) / DAY
    ax.plot(past, s["y"], color="#1f95cc", lw=1.2, label="measured use")
    ax.plot(days_fwd, p, color="#c47a08", lw=1.5, label="TimesFM-3")
    ax.fill_between(days_fwd, q[:, 0], q[:, -1], color="#c47a08", alpha=0.18, lw=0, label="q0.1-q0.9")
    ax.axvline(0, color="#888", lw=0.8, ls=":")
    ax.set_title(f"{s['stationCode'].title()} · {s['item']} ({s['unit']}/day)", fontsize=10)
    ax.grid(alpha=0.25)
for ax in list(axes.flat)[len(series):]:
    ax.axis("off")
axes[0][0].legend(fontsize=8, loc="upper left")
fig.suptitle("Daily consumption: 60 days measured (simulated) + TimesFM-3 forecast", fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
doc = {"model": MODEL_ID, "generatedAt": int(time.time() * 1000), "today": TODAY,
       "horizon": HORIZON, "rows": rows, "summary": summary}
with open(OUT_FILE, "w") as f:
    json.dump(doc, f)
print(f"Wrote {len(rows)} forecasts to {OUT_FILE}")

if CONVEX_URL and PUSH_TO_CONVEX:
    n = convex("mutation", "forecast:importTimesfm", {"rows": rows})
    print(f"Imported {n} forecasts into {CONVEX_URL}. Open /inventory to see TimesFM-3 on the burn-down.")
else:
    try:
        from google.colab import files
        files.download(OUT_FILE)
    except ImportError:
        pass
    print(f"Next: npm run forecast:import -- {OUT_FILE}")